<p> <center> <a href="../../Start-NIM-RAG.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="rag_nim_endpoints.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="rag_nim_endpoints.ipynb">1</a>
        <a >2</a>
        <a href="nim_lora_adapter.ipynb">3</a>
        <!-- <a href="challenge.ipynb">4</a> -->
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="nim_lora_adapter.ipynb">Next Notebook</a></span>
</div>

# ローカライズされたNVIDIA NIMによるRAGの構築
---


このNotebookでは、ローカライズされたNVIDIA NIM(NVIDIA Inference Microservice)を使用してRAG (検索拡張生成)パイプラインを構築するデモンストレーションを行います。まず初めに、NVIDIA API Keyの設定、NIMイメージの取得とデプロイ、ローカルにデプロイされたNIMを使用するRAGアプリケーションの構築について説明します。

### NVIDIA API キーの設定

前回のNotebookでは、生成したNVIDIA API KEYの設定方法を学びました。このNotebookの要件として、NVIDIA NIM の docker イメージを取得する為に、環境変数 `NVIDIA_API_KEY` としてキーを設定する必要があります。まだキーを取得していない場合は、NVIDIA NIM API [ホームページ](https://build.nvidia.com/explore/discover) にアクセスしてAPIキーを生成してください。以下のセルを実行し、表示されるテキストボックスにNVIDIA API KEYを入力し、キーボードのEnterキーを押してください。

In [1]:
import os
import getpass

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key


Enter your NVIDIA API key:  ········


### セルフホスト型NIM

以下のセルを実行して、dockerデーモンが起動していることを確認してください。

In [2]:
! docker ps

CONTAINER ID   IMAGE                                               COMMAND                  CREATED        STATUS                 PORTS                                                                                      NAMES
f9e2c84cfaef   zilliz/attu:v2.3.5                                  "docker-entrypoint.s…"   4 months ago   Up 4 weeks             0.0.0.0:3001->3000/tcp, :::3001->3000/tcp                                                  milvus-attu
e080d67bbdaa   milvusdb/milvus:v2.4.17-gpu                         "/tini -- milvus run…"   4 months ago   Up 4 weeks (healthy)   0.0.0.0:9091->9091/tcp, :::9091->9091/tcp, 0.0.0.0:19530->19530/tcp, :::19530->19530/tcp   milvus-standalone
971ba8ea9b78   minio/minio:RELEASE.2023-03-20T20-16-18Z            "/usr/bin/docker-ent…"   4 months ago   Up 4 weeks (healthy)   0.0.0.0:9000-9001->9000-9001/tcp, :::9000-9001->9000-9001/tcp                              minio
f840a4024587   quay.io/coreos/etcd:v3.5.5                          "etcd -

**期待される出力 (実行中のコンテナが存在しない場合):**

```python

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES

```

### NVCR (NVIDIA Container Registry)へのログイン

NVIDIA NIM の docker イメージにアクセスするには、`docker login nvcr.io`でログインする必要があります。このプロセスでは、`--username $oauthtoken`としてのデフォルトのユーザ名と、`$NGC_API_KEY`の値を受け付ける `--password-stdin` が必要です。

In [3]:
! echo -e "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin

WARNING! Your password will be stored unencrypted in /home/mmurakami/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded


**想定される出力**:
```
WARNING! Your password will be stored unencrypted in /home/yagupta/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
```

### NVIDIA NIMの選択

[NIMs Catalog](https://build.nvidia.com/explore/reasoning)には、さまざまなドメインにおける複数の最先端モデルが掲載されています。下のスクリーンショットのように、`RUN ANYWHERE`タグの付いたものを探してください。これらのNIMイメージはダウンロード可能で、モデルや必要な最適化ランタイムを含んでおり、すぐに使い始めるのに役立ちます。

<img src="imgs/catalog1.jpg" style="width: 900px; height: auto;">

お好みのNIMモデルを選択し、dockerタブをクリックし、以下のスクリーンショットのように赤枠内のイメージ名をコピーします。 

<img src="imgs/catalog2.jpg" style="width: 900px; height: auto;">

### ImageのPull

次のステップはDockerイメージをPullすることです。ここでは、`llama3-3.1-swallow-8b-instruct-v0.1`をプルすることでこのステップを実演します。

In [4]:
! docker pull nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest

latest: Pulling from nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1
Digest: sha256:00dbe414f31d19cb63b09795d40e74bb309456b9cb32f580110a20e4473849d6
Status: Image is up to date for nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest
nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest


**想定される出力:** (すでにImageを取得している場合)
```python

latest: Pulling from nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1
Digest: sha256:00dbe414f31d19cb63b09795d40e74bb309456b9cb32f580110a20e4473849d6
Status: Image is up to date for nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest
nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest
```

利用可能なイメージをリストアップして、モデルイメージをチェックしてみましょう。*`IMAGE ID`は、以下の期待される出力の下に表示されるものと異なる可能性があることに注意してください*。

In [ ]:
! docker image ls

**想定される出力**:

```python
REPOSITORY         TAG       IMAGE ID       CREATED        SIZE
nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1     latest     3882ef11dbcc   5 months ago   18.9GB
```


#### モデル アーティファクトの為のキャッシュの設定

NVIDIA NIMは、ハードウェア上で最大のパフォーマンスを達成するために最適なプロファイルが選択されるように、多くのファイルをダウンロードします。モデル成果物をキャッシュする場所を `LOCAL_NIM_CACHE` として設定し、その変数をエクスポートします。

In [6]:
from os.path import expanduser
home = expanduser("~")
os.environ['LOCAL_NIM_CACHE']=f"{home}/.cache/nim"
!echo $LOCAL_NIM_CACHE

/home/mmurakami/.cache/nim


In [7]:
!mkdir -p "$LOCAL_NIM_CACHE"
# !chmod 777 "$LOCAL_NIM_CACHE"

### NIM LLM マイクロサービスの起動

以下のセルを実行し docker run コマンドを実行して NIM LLM マイクロサービスを起動します。

```python
docker run -it --rm -d --gpus all --name=llm_nim --shm-size=16GB  -e NGC_API_KEY  -v '$LOCAL_NIM_CACHE':/opt/nim/.cache  -u $(id -u) -p 8000:8000  nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1
```

このDockerコマンドは、以下のフラグを使ってNIM LLMマイクロサービスを起動します:

- `-it`: 擬似TTYを割り当て、対話プロセス用にSTDINをオープンしておきます
- `--rm`:  終了時にコンテナを自動的に削除します
- `-d`: コンテナをデタッチモード（バックグラウンド）で実行します
- `--gpus all`: コンテナをデタッチモード（バックグラウンド）で実行する： コンテナが利用可能なすべての GPU にアクセスできるようにします
- `--name=llm_nim`: コンテナに"llm_nim"という名前をつけます
- `--shm-size=16GB`: `/dev/shmの`サイズを 16GB に設定します
- `-e NGC_API_KEY`: 環境変数 NGC_API_KEY をコンテナに渡します
- `-v $LOCAL_NIM_CACHE:/opt/nim/.cache`: ローカル NIM キャッシュディレクトリをコンテナの /opt/nim/.cache にマウントします
- `-u $(id -u)`: 現在のユーザーの UID でコンテナを実行します
- `-p 8000:8000`: ホストのポート 8000 をコンテナのポート 8000 にマップします
- `nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1`: 使用するDockerイメージを指定します

システムは複数のプロセスを実行することができるので、実行中のアプリケーションでポートを占有しないようにしなければなりません。以下のコードでは、一意の空きポートを見つけて割り当てています：

In [8]:
import random
import socket

def find_available_port(start=11000, end=11999):
    while True:
        # Randomly select a port between start and end range
        port = random.randint(start, end)
        
        # Try to create a socket and bind to the port
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("localhost", port))
                # If binding is successful, the port is free
                return port
            except OSError:
                # If binding fails, the port is in use, continue to the next iteration
                continue

# Find and print an available port
os.environ['CONTAINER_PORT'] = str(find_available_port())
print(f"Your have been alloted the available port: {os.environ['CONTAINER_PORT']}")

Your have been alloted the available port: 11587


In [9]:
! docker run -it -d --rm \
--gpus all \
--name=llm_nim \
--shm-size=16GB  \
-e $NGC_API_KEY \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-u $(id -u) \
-p $CONTAINER_PORT:8000 \
nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest

# In order to ensure, the local NIM container is completely loaded and doesn't remain in pending stage, we instantiate a wait interval
! sleep 60

c9c05e2abff2e85fe277564a634d56ba22ceed32fd0be4b595eaa8c9c282d11c


In [13]:
! docker logs --tail 45 llm_nim

Error response from daemon: No such container: llm_nim


**想定される出力:**
```
0.0.0.0:8000/v1/metrics
  0.0.0.0:8000/v1/health/ready
  0.0.0.0:8000/v1/health/live
  0.0.0.0:8000/v1/models
  0.0.0.0:8000/v1/license
  0.0.0.0:8000/v1/metadata
  0.0.0.0:8000/v1/manifest
  0.0.0.0:8000/v1/version
  0.0.0.0:8000/v1/chat/completions
  0.0.0.0:8000/v1/completions
  0.0.0.0:8000/experimental/ls/inference/chat_completion
  0.0.0.0:8000/experimental/ls/inference/completion
INFO 2025-05-26 07:20:10.123 api_server.py:663] An example cURL request:
curl -X 'POST' \
  'http://0.0.0.0:8000/v1/chat/completions' \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{
    "model": "tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1",
    "messages": [
      {
        "role":"user",
        "content":"Hello! How are you?"
      },
      {
...
INFO 2025-05-26 07:20:10.151 server.py:82] Started server process [66]
INFO 2025-05-26 07:20:10.152 on.py:48] Waiting for application startup.
INFO 2025-05-26 07:20:10.154 on.py:62] Application startup complete.
INFO 2025-05-26 07:20:10.155 server.py:214] Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)

```

### クイックテストの開始
NIMが稼働していることを2つの方法で素早くテストできます:
- LangChain NVIDIA Endpoints
- 単純なOpenAI完了リクエスト

**パラメータの説明:**
- **base_url**:  NVIDIA NIM の docker イメージがデプロイされているURL
- **model**: デプロイされた NVIDIA NIM モデルの名前
- **temperature**: サンプリングのランダム性を調整する。温度を下げると、高い確率で単語が選択される可能性が高くなります。
- **top_p**: モデルがどの程度決定論的かを制御します。正確で事実に基づいた回答を求める場合は、この値を低く保ちます。より多様な回答を求める場合は、値を大きくします
- **max_tokens**: 生成される出力トークンの最大数


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']), model="tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", temperature=0.1, max_tokens=1000, top_p=1.0)

result = llm.invoke("フランスの首都はどこですか？")
print(result.content)

エラーが出力された場合は、しばらく待ってから上記のセルを再実行してください。エラーは、NIMコンテナが完全に立ち上がっていないことが原因かもしれません。

In [ ]:
!curl -X 'POST' \
    "http://0.0.0.0:${CONTAINER_PORT}/v1/completions" \
    -H "accept: application/json" \
    -H "Content-Type: application/json" \
    -d '{"model": "tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", "prompt": "フランスの首都はどこですか?", "temperature": 0.1, "top_p": 1.0, "max_tokens": 64}'

### RAG アプリケーション

このセクションでは、前のノートブックの手順に従って、ローカルに配置されたNVIDIA NIMをベースとしたRAGアプリケーションを構築します。このデモでは、前のノートブックのように2つのLLMを使って会話型検索チェーンを作るのではなく、1つのLLM `llama-3.1-swallow--8b-instruct` を使って会話型検索チェーンを作ります。これは各NIMイメージには1つのベースモデルがあるからです。ローカルに配置されたNIMとリモートアクセスを使用することも可能ですが、わかりやすくするために、単一のLLMのアプローチにこだわります。 

#### ライブラリのインポート

In [ ]:
from langchain.chains import ConversationalRetrievalChain, LLMChain
from langchain.chains.conversational_retrieval.prompts import CONDENSE_QUESTION_PROMPT, QA_PROMPT
from langchain.chains.question_answering import load_qa_chain
from langchain.memory import ConversationBufferMemory
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

#### ウェブリンク・データソースの作成

お好みのウェブリンクを差し替えたり、追加したりすることができます。

In [ ]:
urls = [
"https://www.fsa.go.jp/news/r5/20230829/230829_main.pdf",
"https://www.tioj.or.jp/activity/pdf/190619_06.pdf",
"https://www.pmda.go.jp/files/000251332.pdf",
"https://www.jili.or.jp/files/research/zenkokujittai/pdf/r3/i-xvii.pdf",
"https://www.zenginkyo.or.jp/fileadmin/res/news/news350331_1.pdf"
]


## 実装

In [ ]:
!uv pip install pypdf

#### PDF ファイルを読み込む関数を作る

以下は、PDFファイルを読み込むためのヘルパー関数です。

In [ ]:
from pypdf import PdfReader
from io import BytesIO
import requests
import re

def clean_japanese_pdf_text(text: str) -> str:
    # 日本語文中の改行を削除
    text = re.sub(r'(?<=[ぁ-んァ-ヶ一-龥])\s*\n\s*(?=[ぁ-んァ-ヶ一-龥])', '', text)
    # 連続するスペースや全角スペースを1つにまとめる
    text = re.sub(r'[ 　]{2,}', ' ', text)
    # 各行の先頭空白を削除
    text = re.sub(r'^\s+', '', text, flags=re.MULTILINE)
    # 「..... 4」形式の目次行のページ番号を除去
    text = re.sub(r'\.{3,}\s*\d{1,3}', '', text)
    text = text.replace("\n", "")
    return text

def pdf_document_loader(url: str) -> str:
    """
    Loads and extracts cleaned text from a PDF at the given URL using `pypdf`.

    Args:
        url: The URL of the PDF.

    Returns:
        Cleaned text content of the PDF.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        reader = PdfReader(BytesIO(response.content))
        text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted
        return clean_japanese_pdf_text(text.strip())
    except Exception as e:
        print(f"Failed to load {url} due to: {e}")
        return ""



#### 埋め込みとDocument Text Splitterの作成

埋め込みを保存するパスを初期化し、`pdf_document_loader`関数を実行し、ドキュメントをテキストの塊に分割する関数を作ってみましょう。

In [ ]:
from tqdm import tqdm

def create_embeddings(embeddings_model,embedding_path: str = "./embed"):

    embedding_path = "./embed"
    print(f"Storing embeddings to {embedding_path}")

    documents = []
    for url in tqdm(urls):
        print(url)
        document = pdf_document_loader(url)
        #document = html_document_loader(url)
        documents.append(document)


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=0,
        length_function=len,
    )
    print("Total documents:",len(documents))
    texts = text_splitter.create_documents(documents)
    print("Total texts:",len(texts))
    index_docs(embeddings_model,url, text_splitter, texts, embedding_path,)
    print("Generated embedding successfully")

#### LangChainからNVIDIA AI Endpointsを使って埋め込みを生成する

このセクションでは、LangChain用のNVIDIA AI Endpointsを使って埋め込みを生成し、将来の再利用のためにエンベッディングを`/embed`ディレクトリのオフラインベクタストアに保存する方法をデモします。

In [ ]:
embeddings_model = NVIDIAEmbeddings(model="NV-Embed-QA") # or use nvidia/nv-embedqa-e5-v5

以下では、ドキュメントページのコンテンツをループしてテキストとメタデータを拡張し、[FAISS](https://faiss.ai/index.html)を適用する `index_docs` 関数を作成します。埋め込みはローカルに保存されます。

In [ ]:
from typing import List, Union


def index_docs(embeddings_model, url: Union[str, bytes], splitter, documents: List[str], dest_embed_dir: str) -> None:
    """
    Split the documents into chunks and create embeddings for them.
    
    Args:
        embeddings_model: Model used for creating embeddings.
        url: Source url for the documents.
        splitter: Splitter used to split the documents.
        documents: List of documents whose embeddings need to be created.
        dest_embed_dir: Destination directory for embeddings.
    """
    texts = []
    metadatas = []

    for document in documents:
        chunk_texts = splitter.split_text(document.page_content)
        texts.extend(chunk_texts)
        metadatas.extend([document.metadata] * len(chunk_texts))

    if os.path.exists(dest_embed_dir):
        docsearch = FAISS.load_local(
            folder_path=dest_embed_dir, 
            embeddings=embeddings_model, 
            allow_dangerous_deserialization=True
        )
        docsearch.add_texts(texts, metadatas=metadatas)
    else:
        docsearch = FAISS.from_texts(texts, embedding=embeddings_model, metadatas=metadatas)

    docsearch.save_local(folder_path=dest_embed_dir)

#### ベクターストアからの埋め込みのロードとNVIDIA Endpointを使用したRAGの構築

次に、関数 `create_embeddings` を呼び出し、FAISSを使って[vector store](https://developer.nvidia.com/blog/accelerating-vector-search-fine-tuning-gpu-index-algorithms/)から文書を読み込む。ベクトルストアはembeddingsと呼ばれる高次元空間に関連情報を格納しています。

以下の2つのセルを実行してください。

In [ ]:
%%time
create_embeddings(embeddings_model=embeddings_model)

In [ ]:
# load Embed documents
embedding_path = "./embed/"
docsearch = FAISS.load_local(folder_path=embedding_path, embeddings=embeddings_model, allow_dangerous_deserialization=True)

### llama-3.1-swallow-8b-instructで会話型検索チェーンを作る

デプロイされたNIMは`http://0.0.0.0:8000`で稼働しているので、NIMの基本モデル`llama-3.1-swallow-8b-instruct`に基づいて[会話型検索チェーン](https://python.langchain.com/v0.1/docs/modules/chains/#conversationalretrievalchain-with-streaming-to-stdout)を作成する。

In [ ]:
llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']),
                 model="tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", temperature=0.0, max_tokens=1000, top_p=1.0)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

qa_prompt=QA_PROMPT

doc_chain = load_qa_chain(llm, chain_type="stuff", prompt=QA_PROMPT)

qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=docsearch.as_retriever(),
    chain_type="stuff",
    memory=memory,
    combine_docs_chain_kwargs={'prompt': qa_prompt},
)

### クエリによるテスト

In [ ]:
query = "世帯主が不意の事故により入院が必要になる場合の必要資金について、60～64歳及び65歳以上の夫婦が公的年金以 \
        外に必要とする月間生活資金と比較してください。"
result = qa({"question": query})
print(result.get("answer"))

In [ ]:
query = "製造たばこの個包装及び中間包装に求められる識別マークの表示方法の条件について説明してください。"
result = qa({"question": query})
print(result.get("answer"))

先に進む前に、dockerコンテナを停止してGPUのVRAMを解放しましょう。

In [ ]:
! docker container stop llm_nim

次のノートブックでは、LoRAのようなPEFTの機能をNIMに追加する方法を説明します。

このノートブックではNVIDIA NIMとLangChainを用いてRAGアプリケーションの作成を行いました。
以下のオプショナル演習は、NVIDIA NIMの機能に関する追加演習です。

### オプション演習1 NVIDIA NIMの環境変数について学ぶ
NVIDIA NIMはあらかじめビルド済のプロファイル、 NIMコンテナをpullしたサーバ上でのjust-in-timeコンパイルのオプションなど、各NIM毎にいくつかのプロファイルが用意されています。
vLLMエンジンで明示的に起動したい。Tensor Parallel数を明示的に指定したい等、推論サーバの設定をカスタムしたい場合の為に、以下でNVIDIA NIMのプロファイルについて学びます。

以下のセルを実行しllama-3.1-swallow-8b-instruct-v0.1のプロファイル一覧を取得します。

In [ ]:
! docker run -it --rm \
--gpus '"device=1"' \
--name=llm_nim \
--shm-size=16GB  \
-e $NGC_API_KEY \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-u $(id -u) \
-p $CONTAINER_PORT:8000 \
nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest list-model-profiles

**想定される出力:**
```
SYSTEM INFO
- Free GPUs:
  -  [20b5:10de] (0) NVIDIA A100 80GB PCIe [current utilization: 1%]
Profile 375dc0ff86133c2a423fbe9ef46d8fdf12d6403b3caa3b8e70d7851a89fc90dd is not fully defined with checksums
Profile 4f904d571fe60ff24695b5ee2aa42da58cb460787a968f1e8a09f5a7e862728d is not fully defined with checksums
Profile 54946b08b79ecf9e7f2d5c000234bf2cce19c8fee21b243c1a084b03897e8c95 is not fully defined with checksums
Profile 7fa4a5a68c0338f16aef61de94977acfdacb7cabd848d38c49c48d2f639f04b3 is not fully defined with checksums
Profile ac34857f8dcbd174ad524974248f2faf271bd2a0355643b2cf1490d0fe7787c2 is not fully defined with checksums
Profile c84b2a068e56a551906563035ed77f88c88cbe1a63c6768fb2d4a9e0af1e67ba is not fully defined with checksums
MODEL PROFILES
- Compatible with system and runnable:
  - 4f904d571fe60ff24695b5ee2aa42da58cb460787a968f1e8a09f5a7e862728d (vllm-bf16-tp1-pp1)
  - With LoRA support:
- Compilable to TRT-LLM using just-in-time compilation of HF models to TRTLLM engines:
  - ac34857f8dcbd174ad524974248f2faf271bd2a0355643b2cf1490d0fe7787c2 (tensorrt_llm-trtllm_buildable-bf16-tp1-pp1)
  - With LoRA support:
- Incompatible with system:
  - c84b2a068e56a551906563035ed77f88c88cbe1a63c6768fb2d4a9e0af1e67ba (vllm-bf16-tp4-pp1)
  - 7fa4a5a68c0338f16aef61de94977acfdacb7cabd848d38c49c48d2f639f04b3 (vllm-bf16-tp2-pp1)
  - 54946b08b79ecf9e7f2d5c000234bf2cce19c8fee21b243c1a084b03897e8c95 (tensorrt_llm-trtllm_buildable-bf16-tp4-pp1)
  - 375dc0ff86133c2a423fbe9ef46d8fdf12d6403b3caa3b8e70d7851a89fc90dd (tensorrt_llm-trtllm_buildable-bf16-tp2-pp1)
```


出力の中に`Compatible with system and runnable`、`Compilable to TRT-LLM using just-in-time compilation`、`Incompatible with system`がある事を確認して下さい。
`Compatible with system and runnable`で表示されているのが、NVIDIA NIMのあらかじめビルド済のプロファイルの中で実行環境上で実行可能なプロファイル一覧、`Compilable to TRT-LLM using just-in-time compilation`が実行環境上でjust-in-timeすることで実行可能なプロファイルになります。
`NIM_MODEL_PROFILE`環境変数を用いる事で、プロファイルを明示的に指定したNVIDIA NIMの起動が可能です。
以下では`vllm-bf16-tp1-pp1`というビルド済のプロファイルを明示的に指定した起動の例です。

In [ ]:
! docker run -it --rm \
--gpus '"device=1"' \
--name=llm_nim \
--shm-size=16GB  \
-e $NGC_API_KEY \
-e NIM_MODEL_PROFILE=vllm-bf16-tp1-pp1 \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-u $(id -u) \
-p $CONTAINER_PORT:8000 \
nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1:latest

NVIDIA NIMの環境変数に関する情報は[こちら](https://docs.nvidia.com/nim/large-language-models/latest/configuration.html#environment-variables)にあります。今回紹介した`NIM_MODEL_PROFILE`以外にも細かな制御を行う為の様々なオプションが用意でされています。様々なフラグを有効化した起動をぜひお試しください

### オプション演習2 NVIDIA NIMのファインチューニングモデルサポート機能を使用して最新のLlama3.1-Swallowモデルのデプロイを行う
バージョン1.3以降のNVIDIA NIMでは[ファインチューニングモデルサポート機能](https://docs.nvidia.com/nim/large-language-models/latest/ft-support.html)が追加されました。NVIDIA NIMのファインチューニング機能を使用すると、モデルの重みをファインチューニング後の重みに変更して推論サーバーを起動する事が可能です。
ファインチューニングモデルサポート機能については[エヌビディア技術ブログ](https://developer.nvidia.com/ja-jp/blog/deploying-fine-tuned-ai-models-with-nvidia-nim/)にも記事が公開されています。併せてご確認下さい。
例えば、[Llama-3.1-Swallow](https://swallow-llm.github.io/llama3.1-swallow.ja.html)は[Meta社のLlama-3.1](https://huggingface.co/collections/meta-llama/llama-31-669fc079a0c406a149a5738f)に対してファインチューニングを実施し作成されたモデルです。つまり、Meta Llama-3.1-8BのNIMに対してファインチューニングモデル機能を使用する事で、Llama-3.1-Swallow-8Bの重みに変更して推論サーバーを起動する事が可能です。

[NVIDIA APIカタログ](https://build.nvidia.com/)に公開されているNIM化されたモデルは常にHuggingFaceに公開されているモデル情報より古い可能性があります。例えば[NIM化済のLlama-3.1-Swallow-8B-Instruct](https://catalog.ngc.nvidia.com/orgs/nim/teams/tokyotech-llm/containers/llama-3.1-swallow-8b-instruct-v0.1)は、[こちらの重み](https://huggingface.co/tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.1)を使ってNIM化されています。
しかしLlama-3.1-Swallow-8B-Instructは定期的に更新されており、重みが更新された[Llama-3.1-8B-Instruct-v0.5](https://huggingface.co/tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.5)が最新です。

以下を実行し、NIMのファインチューニング機能を使用したLlama-3.1-8B-Instruct-v0.5のNIMの起動を試してみましょう。

まずはMeta Llama-3.1-8B-InstructのNIMコンテナをpullしましょう。

In [ ]:
!docker pull nvcr.io/nim/meta/llama-3.1-8b-instruct:1.8.4

In [ ]:
import os
from huggingface_hub import snapshot_download
 
MODEL_DIR = "./models"
os.makedirs(MODEL_DIR, exist_ok=True)
 
snapshot_download(
    repo_id="tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.5",
    local_dir=f"{MODEL_DIR}/Llama-3.1-Swallow-8B-Instruct-v0.5",
    local_dir_use_symlinks=False
    )

In [63]:
current_path = os.getcwd()
os.environ['NIM_SERVED_MODEL_NAME']="Llama-3.1-Swallow-8B-v0.5"
os.environ['TRTLLM_MODEL_PATH']=current_path+"/models/Llama-3.1-Swallow-8B-Instruct-v0.5"

In [70]:
!docker run -it -d --rm \
--name=swallow \
--gpus '"device=1"' \
--shm-size=16GB \
-e NIM_MODEL_NAME=/model \
-e NIM_SERVED_MODEL_NAME=${NIM_SERVED_MODEL_NAME} \
-e NIM_MODEL_PROFILE=tensorrt_llm \
-e NIM_ENABLE_KV_CACHE_REUSE=1 \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-v $TRTLLM_MODEL_PATH:/model \
-u $(id -u) \
-p $CONTAINER_PORT:8000 \
nvcr.io/nim/meta/llama-3.1-8b-instruct:1.8.4

94f35f7f56ff7492d13929c7c07935989c018ddd9baa74bc9e4d7812aef0028e


NIMをデプロイしたサーバ上でjust-in-timeコンパイルが実行され、TensorRT-LLMエンジンがビルドされます。ビルド後に推論サーバーが起動します。
ビルドには暫く時間がかかります。
数分したら、以下のセルを実行しサーバが起動済が確認してみましょう。

In [ ]:
! docker logs --tail 100 swallow

以下のセルを実行してクエリーを推論サーバに投げ、応答が返ってくるかテストしてみましょう。

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']), model="tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", temperature=0.1, max_tokens=1000, top_p=1.0)

result = llm.invoke("東京の紅葉した公園で、東京タワーと高層ビルを背景に、空を舞うツバメと草地に佇むラマが出会う温かな物語を書いてください。")
print(result.content)

In [ ]:
! docker container stop swallow

### オプション演習3 Model-less LLM NIMの機能を試す
最近リリースされたバージョン0.11以降のNVIDIA NIMコンテナは、機能アップデートにより更に柔軟になりました。`TensorRT-LLM`および`vLLM`のサポートに加えて`SGLang`のサポートも追加されました。また既存のファインチューニングモデルサポート機能では、同一アーキテクチャのNIM化されたモデルコンテナを用意する必要がありましたが、バージョン0.11で追加されたModel-less LLM NIMの機能により、その制約がなくなりました。
つまり、[NVIDIA APIカタログ](https://build.nvidia.com/)に公開されていないモデルであっても、HuggingFace等にモデル情報が公開されていれば、just-in-timeコンパイルで起動時に推論エンジンをビルドし、NIM化できるようになりました。

例えば[phi-4](https://huggingface.co/microsoft/phi-4)のNIM化モデルはまだ[NVIDIA APIカタログ](https://build.nvidia.com/)に公開されていません。Model-less LLM NIMの機能を使用して、phi-4の推論サーバーを起動してみましょう。

以下のセルを実行し、Model-less LLM NIM機能の追加された新しいNIVIDIA NIMコンテナをpullします。

In [ ]:
!docker pull nvcr.io/nvidian/nim-llm-dev/universal-nim:1.11.0.rc2

次にphi-4をHuggingFaceからダウンロードします。

In [ ]:
snapshot_download(
    repo_id="microsoft/Phi-4-reasoning",
    local_dir=f"{MODEL_DIR}/Phi-4-reasoning",
    local_dir_use_symlinks=False
    )

以下のセルを実行し、phi-4のNIM推論サーバを起動します。

In [89]:
current_path = os.getcwd()
os.environ['NIM_SERVED_MODEL_NAME']="Phi-4-reasoning"
os.environ['TRTLLM_MODEL_PATH']=current_path+"/models/Phi-4-reasoning"

In [90]:
!docker run -it --rm \
--gpus '"device=1"' \
--shm-size=16GB \
-e NIM_MODEL_NAME=/model \
-e NIM_SERVED_MODEL_NAME=${NIM_SERVED_MODEL_NAME} \
-e NIM_MODEL_PROFILE=vllm \
-e NIM_ENABLE_KV_CACHE_REUSE=1 \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-v $TRTLLM_MODEL_PATH:/model \
-u $(id -u) \
-p $CONTAINER_PORT:8000 \
nvcr.io/nvidian/nim-llm-dev/universal-nim:1.11.0.rc2


== NVIDIA Inference Microservice LLM NIM ==

Model: /model

Container image Copyright (c) 2016-2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This NIM container is governed by the NVIDIA AI Product Agreement here:
https://www.nvidia.com/en-us/data-center/products/nvidia-ai-enterprise/eula/.
A copy of this license can be found under /opt/nim/LICENSE.

The use of this model is governed by the AI Foundation Models Community License
here: https://docs.nvidia.com/ai-foundation-models-community-license.pdf.

INFO 07-01 14:48:41 [__init__.py:256] Automatically detected platform cuda.
[1751381330.159625] [bf30856af300:59   :0]          parser.c:2326 UCX  WARN  unused environment variables: UCX_HOME; UCX_DIR (maybe: UCX_TLS?)
[1751381330.159625] [bf30856af300:59   :0]          parser.c:2326 UCX  WARN  (set UCX_WARN_UNUSED_ENV_VARS=n to suppress this warning)
2025-07-01 14:48:51,009 - INFO - flashinfer.jit: Prebuilt kernels not found, using JIT backend
[TensorRT-LLM] TensorRT-LLM 

---

## References

- https://developer.nvidia.com/blog/tips-for-building-a-rag-pipeline-with-nvidia-ai-langchain-ai-endpoints/
- https://nvidia.github.io/GenerativeAIExamples/latest/notebooks/05_RAG_for_HTML_docs_with_Langchain_NVIDIA_AI_Endpoints.html

## Licensing

Copyright © 2024 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.

<br>
<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="rag_nim_endpoints.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="rag_nim_endpoints.ipynb">1</a>
        <a >2</a>
        <a href="nim_lora_adapter.ipynb">3</a>
        <!-- <a href="challenge.ipynb">4</a> -->
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="nim_lora_adapter.ipynb">Next Notebook</a></span>
</div>

<br>
<p> <center> <a href="../../Start-NIM-RAG.ipynb">Home Page</a> </center> </p>